# Study 926 — T+1 — the teardown

The difference-in-difference specification, HAC inference, block-bootstrap CIs, the
window-length sign flip, the placebo-switch-date distribution, the excess-of-cash Sharpe race, the costed turn-of-month overlay and the live synthetic control. Every real number is frozen from `docs/results.md` (return fingerprint `1dea7df6b22f`, as-of 2026-06-30).

## Specification

For each daily outcome `y` and each treated leg *i*, form the difference against the
control, `d_t = y_i,t − y_SPY,t`, restrict to symmetric windows, and estimate

$$d_t = \alpha + \beta \cdot \mathbb{1}[t \geq \text{2024-05-28}] + \varepsilon_t$$

with Newey-West standard errors at bandwidth $\lfloor 4(n/100)^{2/9} \rfloor$. $\beta$ is the DiD. The common market factor cancels inside `d_t`, which is what makes the estimator well-behaved against the 2022–2026 macro path — but it cannot cancel the **structural** difference that EFA's and EEM's overnight window contains the entire foreign cash session.

> 💡 **In plain words** — we are not asking whether EFA behaves differently from SPY. Of course it does. We are asking whether the *size of that difference* changed on one specific day.

In [1]:
R = {'switch': '2024-05-28', 'asof': '2026-06-30', 'fp': '1dea7df6b22f', 'n_rows': 4802, 'pre_start': '2022-04-26', 'pre_end': '2024-05-24', 'post_start': '2024-05-28', 'post_end': '2026-06-30', 'n_each': 524, 'efa_ron': -0.064, 'efa_ron_t': -0.02, 'efa_abson': 1.08, 'efa_abson_t': 0.35, 'efa_share': 0.023, 'efa_share_t': 1.03, 'efa_abscc': 8.668, 'efa_abscc_t': 2.18, 'eem_ron': 4.827, 'eem_ron_t': 1.11, 'eem_abson': 8.281, 'eem_abson_t': 1.87, 'eem_share': -0.017, 'eem_share_t': -0.73, 'eem_abscc': 19.209, 'eem_abscc_t': 3.39, 'iwm_ron': 0.675, 'iwm_ron_t': 0.25, 'iwm_abson': 6.0, 'iwm_abson_t': 2.24, 'iwm_share': 0.016, 'iwm_share_t': 0.78, 'iwm_abscc': 2.656, 'iwm_abscc_t': 0.63, 'efa_share_pre': 0.531, 'efa_share_post': 0.559, 'eem_share_pre': 0.589, 'eem_share_post': 0.578, 'spy_share_pre': 0.415, 'spy_share_post': 0.42, 'spy_abscc_pre': 82.97, 'spy_abscc_post': 69.13, 'efa_abscc_pre': 78.95, 'efa_abscc_post': 73.78, 'eem_abscc_pre': 85.76, 'eem_abscc_post': 91.13, 'ci_efa_share': (-0.023, 0.069), 'ci_efa_abscc': (-0.93, 18.02), 'ci_eem_share': (-0.059, 0.025), 'ci_eem_abscc': (4.64, 34.44), 'efa_abscc_w': ((63, 12.83, 1.76), (126, 8.78, 1.43), (252, -8.17, -1.53), (524, 8.67, 2.18)), 'eem_abscc_w': ((63, 8.94, 1.14), (126, 10.95, 1.22), (252, -10.71, -1.56), (524, 19.21, 3.39)), 'efa_share_w': ((63, 0.137, 2.31), (126, 0.079, 1.97), (252, 0.038, 1.18), (524, 0.023, 1.03)), 'eem_share_w': ((63, 0.003, 0.04), (126, -0.024, -0.47), (252, -0.008, -0.21), (524, -0.017, -0.73)), 'placebo': (('EFA', 'r_on', 0.02, 1.48, 3.71, 88, 88), ('EFA', 'abs_on', 0.35, 2.77, 10.77, 82, 88), ('EFA', 'on_var_share', 1.03, 1.94, 8.48, 69, 88), ('EFA', 'abs_cc', 2.18, 4.97, 10.64, 69, 88), ('EEM', 'r_on', 1.11, 2.21, 5.05, 47, 81), ('EEM', 'abs_on', 1.87, 3.45, 5.21, 59, 81), ('EEM', 'on_var_share', 0.73, 2.78, 6.22, 66, 81), ('EEM', 'abs_cc', 3.39, 6.26, 10.08, 61, 81), ('IWM', 'r_on', 0.25, 1.82, 4.25, 89, 93), ('IWM', 'abs_on', 2.24, 2.63, 10.44, 55, 93), ('IWM', 'on_var_share', 0.78, 1.92, 6.0, 72, 93), ('IWM', 'abs_cc', 0.63, 2.05, 7.14, 76, 93)), 'sharpe': (('SPY', 0.52, 0.9, 0.34), ('IWM', 0.13, 0.8, 0.66), ('EEM', 0.06, 1.09, 1.14), ('EFA', 0.42, 0.79, 0.36)), 'efa_tom_gross': (-1.73, 5.38), 'efa_tom_net': (-3.42, 3.68), 'efa_tom_did': 7.11, 'efa_tom_did_t': 0.57, 'efa_tom_post_t': 0.4, 'efa_tom_sharpe': 0.17, 'eem_tom_gross': (10.75, 17.8), 'eem_tom_net': (9.05, 16.11), 'eem_tom_did': 7.06, 'eem_tom_did_t': 0.43, 'eem_tom_post_t': 1.4, 'eem_tom_sharpe': 0.96, 'tom_on_days': 100, 'sweep': ((0.0, 0.0, 17.8, 1.55), (3.0, 50.0, 16.11, 1.4), (10.0, 100.0, 12.41, 1.08)), 'syn_pl_ron': 16.47, 'syn_pl_ron_t': 2.73, 'syn_pl_abson': 13.94, 'syn_pl_abson_t': 3.75, 'syn_nl_ron': 1.05, 'syn_nl_ron_t': 0.19, 'syn_nl_abson': 1.06, 'syn_nl_abson_t': 0.31}

## The four outcomes, three legs

In [2]:
rows = [('EFA','r_on',R['efa_ron'],R['efa_ron_t']),
        ('EFA','abs_on',R['efa_abson'],R['efa_abson_t']),
        ('EFA','on_var_share',R['efa_share'],R['efa_share_t']),
        ('EFA','abs_cc',R['efa_abscc'],R['efa_abscc_t']),
        ('EEM','r_on',R['eem_ron'],R['eem_ron_t']),
        ('EEM','abs_on',R['eem_abson'],R['eem_abson_t']),
        ('EEM','on_var_share',R['eem_share'],R['eem_share_t']),
        ('EEM','abs_cc',R['eem_abscc'],R['eem_abscc_t']),
        ('IWM*','r_on',R['iwm_ron'],R['iwm_ron_t']),
        ('IWM*','abs_on',R['iwm_abson'],R['iwm_abson_t']),
        ('IWM*','on_var_share',R['iwm_share'],R['iwm_share_t']),
        ('IWM*','abs_cc',R['iwm_abscc'],R['iwm_abscc_t'])]
print('leg   outcome           DiD    HAC t   flag')
for leg, oc, did, t in rows:
    flag = '<-- |t| > 2' if abs(t) >= 2 else ''
    print('%-5s %-13s %+8.3f %+7.2f   %s' % (leg, oc, did, t, flag))
print('\n* IWM is the DOMESTIC placebo-treated leg: shares and holdings both T+1.')
print('  windows: %s -> %s vs %s -> %s, %d trading days each'
      % (R['pre_start'], R['pre_end'], R['post_start'], R['post_end'], R['n_each']))

leg   outcome           DiD    HAC t   flag
EFA   r_on            -0.064   -0.02   
EFA   abs_on          +1.080   +0.35   
EFA   on_var_share    +0.023   +1.03   
EFA   abs_cc          +8.668   +2.18   <-- |t| > 2
EEM   r_on            +4.827   +1.11   
EEM   abs_on          +8.281   +1.87   
EEM   on_var_share    -0.017   -0.73   
EEM   abs_cc         +19.209   +3.39   <-- |t| > 2
IWM*  r_on            +0.675   +0.25   
IWM*  abs_on          +6.000   +2.24   <-- |t| > 2
IWM*  on_var_share    +0.016   +0.78   
IWM*  abs_cc          +2.656   +0.63   

* IWM is the DOMESTIC placebo-treated leg: shares and holdings both T+1.
  windows: 2022-04-26 -> 2024-05-24 vs 2024-05-28 -> 2026-06-30, 524 trading days each


Three points. First, `on_var_share` — the outcome with the clearest mechanical link to
a settlement change — is insignificant on both treated legs and **carries opposite
signs**. Second, `abs_cc` fires on both treated legs, but the driver is the control:
SPY's mean absolute daily move fell 82.97 → 69.13 bps while EFA's fell 78.95 → 73.78.
Third, the placebo leg fires on `abs_on`, which is a direct falsification of the
settlement interpretation.

## Block-bootstrap CIs on the DiD (2,000 draws, 21-day circular blocks)

Pre and post halves are resampled separately in blocks, and the statistic recomputed
as `mean(post) − mean(pre)`.

In [3]:
for tag, pt, ci in [('EFA on_var_share', R['efa_share'], R['ci_efa_share']),
                    ('EFA abs_cc     ', R['efa_abscc'], R['ci_efa_abscc']),
                    ('EEM on_var_share', R['eem_share'], R['ci_eem_share']),
                    ('EEM abs_cc     ', R['eem_abscc'], R['ci_eem_abscc'])]:
    zero = 'includes 0' if ci[0] <= 0 <= ci[1] else 'excludes 0'
    print('%s  point %+8.3f  95%% CI [%+8.3f, %+8.3f]  %s' % (tag, pt, ci[0], ci[1], zero))

EFA on_var_share  point   +0.023  95% CI [  -0.023,   +0.069]  includes 0
EFA abs_cc       point   +8.668  95% CI [  -0.930,  +18.020]  includes 0
EEM on_var_share  point   -0.017  95% CI [  -0.059,   +0.025]  includes 0
EEM abs_cc       point  +19.209  95% CI [  +4.640,  +34.440]  excludes 0


## Window-length sweep — the sign flip

An event study has no eras in the buy-and-hold sense; the honest substitute is to
measure the same break over one quarter, six months, one year and two years either
side. A real break survives all four.

In [4]:
for tag, sweep in [('EFA on_var_share', R['efa_share_w']),
                   ('EEM on_var_share', R['eem_share_w']),
                   ('EFA abs_cc      ', R['efa_abscc_w']),
                   ('EEM abs_cc      ', R['eem_abscc_w'])]:
    cells = '  '.join('+/-%3dd %+8.3f (t %+5.2f)' % row for row in sweep)
    print('%s  %s' % (tag, cells))
print('\nBoth abs_cc results CHANGE SIGN at a one-year window and change back at two.')

EFA on_var_share  +/- 63d   +0.137 (t +2.31)  +/-126d   +0.079 (t +1.97)  +/-252d   +0.038 (t +1.18)  +/-524d   +0.023 (t +1.03)
EEM on_var_share  +/- 63d   +0.003 (t +0.04)  +/-126d   -0.024 (t -0.47)  +/-252d   -0.008 (t -0.21)  +/-524d   -0.017 (t -0.73)
EFA abs_cc        +/- 63d  +12.830 (t +1.76)  +/-126d   +8.780 (t +1.43)  +/-252d   -8.170 (t -1.53)  +/-524d   +8.670 (t +2.18)
EEM abs_cc        +/- 63d   +8.940 (t +1.14)  +/-126d  +10.950 (t +1.22)  +/-252d  -10.710 (t -1.56)  +/-524d  +19.210 (t +3.39)

Both abs_cc results CHANGE SIGN at a one-year window and change back at two.


## Placebo switch dates — the falsification

Re-run the identical DiD pretending the change happened every 63 trading days instead,
excluding the year either side of the true date so the real break cannot leak into its
own null (Bertrand-Duflo-Mullainathan 2004). Then rank the true |*t*|.

In [5]:
print('leg  outcome        real|t|  median|t|   max|t|   exceeding   percentile')
for leg, oc, real, med, mx, exc, n in R['placebo']:
    pct = 100.0 * (n - exc) / n
    print('%-4s %-13s %7.2f %10.2f %8.2f %8s %11.0f%%'
          % (leg, oc, real, med, mx, '%d / %d' % (exc, n), pct))
print('\nOn every row the MEDIAN fake date produces a larger |t| than the real one.')
print('The headline EEM abs_cc t = +3.39 sits at the 25th percentile of noise.')

leg  outcome        real|t|  median|t|   max|t|   exceeding   percentile
EFA  r_on             0.02       1.48     3.71  88 / 88           0%
EFA  abs_on           0.35       2.77    10.77  82 / 88           7%
EFA  on_var_share     1.03       1.94     8.48  69 / 88          22%
EFA  abs_cc           2.18       4.97    10.64  69 / 88          22%
EEM  r_on             1.11       2.21     5.05  47 / 81          42%
EEM  abs_on           1.87       3.45     5.21  59 / 81          27%
EEM  on_var_share     0.73       2.78     6.22  66 / 81          19%
EEM  abs_cc           3.39       6.26    10.08  61 / 81          25%
IWM  r_on             0.25       1.82     4.25  89 / 93           4%
IWM  abs_on           2.24       2.63    10.44  55 / 93          41%
IWM  on_var_share     0.78       1.92     6.00  72 / 93          23%
IWM  abs_cc           0.63       2.05     7.14  76 / 93          18%

On every row the MEDIAN fake date produces a larger |t| than the real one.
The headline EEM abs_cc

## Excess-of-cash Sharpe race (vs BIL), pre vs post

Both arms excess of the *same* cash leg — which matters here, since the pre-window sits
at ~5% short rates and the post-window walks them down.

In [6]:
print('ETF   exSharpe pre -> post   Welch t')
for tk, pre, post, t in R['sharpe']:
    print('%-4s   %+.2f  ->  %+.2f        %+.2f' % (tk, pre, post, t))
print('\nEverything improved, including the control. That is a bull market, not a')
print('settlement cycle: no Welch t is remotely near 2.')

ETF   exSharpe pre -> post   Welch t
SPY    +0.52  ->  +0.90        +0.34
IWM    +0.13  ->  +0.80        +0.66
EEM    +0.06  ->  +1.09        +1.14
EFA    +0.42  ->  +0.79        +0.36

Everything improved, including the control. That is a bull market, not a
settlement cycle: no Welch t is remotely near 2.


## The costed overlay and its assumption sweep

Dollar-neutral long EFA/EEM, short SPY, on a turn-of-month calendar signal (last
trading day + first three). Position for *t+1* set from the calendar known through *t*
— **one execution lag**, which shifts the *held* window forward: the book is on for the
**first four trading days of each month** and flat on the month-end session itself
(every on-day has within-month rank 0–3). Costs are one-way × NAV on **each** leg at
every position change; the short leg pays borrow. Both are ASSUMPTIONS, not
measurements, so both are swept. The post-period Sharpes below are annualised on a
series that is flat ~80% of days — read the HAC *t*, not the Sharpe.

In [7]:
print('spread                gross pre->post      net pre->post      DiD net (t)   post t')
print('long EFA / short SPY  %+6.2f -> %+6.2f     %+6.2f -> %+6.2f     %+5.2f (%+.2f)   %+.2f'
      % (R['efa_tom_gross'][0], R['efa_tom_gross'][1], R['efa_tom_net'][0],
         R['efa_tom_net'][1], R['efa_tom_did'], R['efa_tom_did_t'], R['efa_tom_post_t']))
print('long EEM / short SPY  %+6.2f -> %+6.2f     %+6.2f -> %+6.2f     %+5.2f (%+.2f)   %+.2f'
      % (R['eem_tom_gross'][0], R['eem_tom_gross'][1], R['eem_tom_net'][0],
         R['eem_tom_net'][1], R['eem_tom_did'], R['eem_tom_did_t'], R['eem_tom_post_t']))
print('\ncost x borrow sweep (post-period net, long EEM / short SPY, bps per on-day):')
for c, b, net, t in R['sweep']:
    print('  cost %5.1f bps, borrow %6.1f bps/yr -> net %+6.2f (t %+.2f)' % (c, b, net, t))
print('\nNever clears |t| = 2, not even gross and borrow-free. The DiD is flat across')
print('the surface because a constant cost cancels in a difference-in-difference.')

spread                gross pre->post      net pre->post      DiD net (t)   post t
long EFA / short SPY   -1.73 ->  +5.38      -3.42 ->  +3.68     +7.11 (+0.57)   +0.40
long EEM / short SPY  +10.75 -> +17.80      +9.05 -> +16.11     +7.06 (+0.43)   +1.40

cost x borrow sweep (post-period net, long EEM / short SPY, bps per on-day):
  cost   0.0 bps, borrow    0.0 bps/yr -> net +17.80 (t +1.55)
  cost   3.0 bps, borrow   50.0 bps/yr -> net +16.11 (t +1.40)
  cost  10.0 bps, borrow  100.0 bps/yr -> net +12.41 (t +1.08)

Never clears |t| = 2, not even gross and borrow-free. The DiD is flat across
the surface because a constant cost cancels in a difference-in-difference.


## Live synthetic control — the estimator is unbiased

A **synthetic** two-fund panel with a planted break (treated leg's overnight drift +8
bps/day, +45 bps on turn-of-month days, +30% idiosyncratic overnight vol, from the
switch onwards) and a null panel where nothing happens. Nothing below reads the real
tape.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from t_plus_one import data, strategy as st
pl_panel, pl_truth = data.synthetic_panel(signal_strength=1.0, seed=926)
pl = st.synthetic_detect(pl_panel, pl_truth)
expected = pl_truth['planted_on_drift_bps'] + pl_truth['planted_tom_bps'] * pl_truth['tom_frac']
print('SYNTHETIC WORLD (never the real tape)')
print('planted overnight drift shift: %+.2f bps expected, %+.2f recovered (t %+.2f)'
      % (expected, pl['did_r_on_bps'], pl['t_r_on']))
print('planted overnight vol lift   : DiD abs_on %+.2f bps (t %+.2f)'
      % (pl['did_abs_on_bps'], pl['t_abs_on']))
nulls = []
for s in range(6):
    p, tr = data.synthetic_panel(signal_strength=0.0, seed=926 + s)
    nulls.append(st.synthetic_detect(p, tr)['t_r_on'])
nulls = np.array(nulls)
print('null x6: t on overnight drift mean %+.2f (sd %.2f), |t| >= 2 in %d/6'
      % (nulls.mean(), nulls.std(ddof=1), int((np.abs(nulls) >= 2).sum())))

SYNTHETIC WORLD (never the real tape)
planted overnight drift shift: +16.31 bps expected, +16.47 recovered (t +2.73)
planted overnight vol lift   : DiD abs_on +13.94 bps (t +3.75)


null x6: t on overnight drift mean -0.53 (sd 0.89), |t| >= 2 in 0/6


## Verdict

- **Signal — None.** The mechanism-linked outcome, `on_var_share`, gives +0.023 (*t* = +1.03) on EFA and -0.017 (*t* = -0.73) on EEM — opposite signs, both insignificant, both block-bootstrap CIs straddling zero. The two |t| ≥ 2 results are `abs_cc` (a total-vol measure whose entire movement is SPY's own compression, and which **flips sign** at a ±252-day window) and `abs_on` on the **domestic placebo** IWM. The placebo-switch-date distribution is decisive: on all twelve reported rows the median arbitrary date exceeds the true one, and the study's largest *t* (+3.39) is beaten by 61 of 81 fake dates whose median is +6.26. The synthetic control recovers a planted break cleanly (+16.47 bps, *t* = +2.73) and is silent on the null (*t* = +0.19), so the miss is the tape's, not the harness's.
- **Tradability — Mirage.** The turn-of-month spread's change at the switch is +7 bps per on-day with HAC *t* of +0.43 (EEM) and +0.57 (EFA), invariant across the whole cost × borrow surface and never clearing |t| = 2 even gross. The EEM spread's post-period Sharpe of +0.96 rests on 100 on-days with its own *t* of +1.40, and was already positive pre-switch.
- **Scope, honestly.** All assumptions are labelled in `docs/results.md`: the 3 bps one-way cost, the 50 bps/yr borrow, the hardcoded event date and the choice of which funds count as treated. The channels where a T+1 effect would actually live — FX swap funding, stock-loan recall timing, ETF creation/redemption fails — are not in a price file, and this study makes no claim about them.